# RAG Context Filtering, Citation Validation and Hit@2

Building a filtered structured context for retrieval, validating LLM citations against retrieved chunks, and computing Hit@2 from RAG trace logs.

_Datasets are not included in this repository._

In [11]:
import pandas as pd

In [12]:
df = pd.read_json('rag_trace_logs.jsonl', lines=True)

In [13]:
df

,trace_id,query,expected_chunk,retrieved_chunks,llm_answer,llm_citations,latency_ms
0,trace_b17b376f,Berapa lama penanganan pasien triase merah?,MED-001#C1,"[MED-001#C1, IT-001#C1]",Pasien Triase Merah harus ditangani maksimal 0...,[MED-001#C1],1200
1,trace_571b705c,Aturan penanganan IGD triase merah?,MED-001#C1,"[MED-OLD#C1, HR-001#C1]",Pasien ditangani maksimal 5 menit setelah admi...,[MED-OLD#C1],1150
2,trace_64165ca0,Uang makan perawat shift malam?,HR-001#C1,"[HR-001#C1, IT-001#C1]",Perawat mendapat ekstra Rp 50.000.,[HR-999#C1],1300
3,trace_5a1cce20,Siapa nama Direktur Utama Rumah Sakit?,None,"[IT-001#C1, HR-001#C1]",Direktur Utama rumah sakit adalah dr. Budi San...,[],2500


In [14]:
df2 = pd.read_csv('nusantaramed_kb.csv')

In [15]:
df2

,doc_id,chunk_id,department,is_active,text
0,MED-001,MED-001#C1,Medis,True,Pasien IGD dengan kategori Triase Merah (Gawat...
1,MED-OLD,MED-OLD#C1,Medis,False,Pasien IGD Triase Merah ditangani maksimal 5 m...
2,HR-001,HR-001#C1,HR,True,Perawat shift malam berhak mendapatkan ekstra ...
3,IT-001,IT-001#C1,IT,True,"Jika sistem rekam medis EMR down, dokter wajib..."


In [61]:
def Filter(departemen):
    xar = df2[df2['department'] == departemen]
    xar = xar[xar['is_active'] == True]
    xar["structured"] = xar.apply(lambda x: f"CHUNK_ID: {x['chunk_id']} | DEPT: {x['department']} Teks: {x['text']}", axis=1)
    return "\n".join(xar["structured"].tolist())

In [53]:
def Validate(llm_citations: list, retrieved_chunks: list):
  fake = [x for x in llm_citations if x not in retrieved_chunks]
  if not fake:
      return "VALID"
  else:
        return "INVALID_CITATION", fake

In [72]:
def calculate(df):
  df_expected = df[df["expected_chunk"].notnull()].copy()
  df_expected["hit"]= df.apply(lambda x: x["expected_chunk"] in x["retrieved_chunks"], axis=1)
  hit_rate = df_expected["hit"].mean()
  return f"{hit_rate*100:.2f}%"

In [62]:
Filter("Medis")

'CHUNK_ID: MED-001#C1 | DEPT: Medis Teks: Pasien IGD dengan kategori Triase Merah (Gawat Darurat) harus mendapatkan penanganan dokter maksimal 0 menit sejak kedatangan.'

In [55]:
print(Validate(["Xom"], ['HR-999#C1', 'IT-001#C1']))

('INVALID_CITATION', ['Xom'])


In [73]:
print(calculate(df))

66.67%
